# 🎬 Are Fandango's Movie Ratings Inflated? (Audited & Production Edition)

In 2015, Walt Hickey (FiveThirtyEight) published an investigation claiming that Fandango's movie star ratings were systematically higher than competitor review sites and their own underlying HTML scores.

This audited notebook runs the full analytical pipeline, performs rigorous hypothesis tests, explores temporal shifts (2015 vs 2016-17), and verifies all quantitative claims.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from src.data_loader import DataLoader
from src.analysis import RatingAnalyzer
from src.statistics import StatisticalEngine
from src.database import DatabaseManager

# Initialize loaders and analytical engines
loader = DataLoader()
analyzer = RatingAnalyzer(loader)
stats_engine = StatisticalEngine(loader)
db = DatabaseManager(loader)

df_comp = loader.load_comparison()
df_scrape = loader.load_scrape()
df_after = loader.load_after()

print(f"Loaded {len(df_comp)} 2015 comparison films, {len(df_scrape)} scrape records, and {len(df_after)} 2016-17 films.")

## 1. Executive Summary & KPIs

In [ ]:
kpis = analyzer.get_kpi_overview()
for k, v in kpis.items():
    print(f"{k:30s}: {v}")

## 2. Statistical Hypothesis Testing
We formally test whether the discrepancy between displayed stars and true HTML rating is statistically significant (Right-Tailed Paired Test).

In [ ]:
inflation_results = stats_engine.test_inflation_significance()
for k, v in inflation_results.items():
    print(f"{k:30s}: {v}")

## 3. Temporal Shift: Did Fandango Correct the Bias in 2016–2017?

In [ ]:
temporal_results = stats_engine.test_temporal_shift_significance()
for k, v in temporal_results.items():
    print(f"{k:30s}: {v}")

## 4. SQL Analytics Console Queries

In [ ]:
query = '''
SELECT
    FILM,
    Fandango_Stars,
    Fandango_Ratingvalue,
    ROUND(Fandango_Stars - Fandango_Ratingvalue, 2) AS diff,
    RT_norm,
    IMDB_norm
FROM fandango_2015
WHERE (Fandango_Stars - Fandango_Ratingvalue) >= 0.4
ORDER BY diff DESC
LIMIT 10;
'''
res = db.execute_query(query)
pd.DataFrame(res["rows"])